In [51]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [52]:
data_path = '/content/data.csv'
data = pd.read_csv(data_path)


In [53]:
# Checking for missing values
if data.isnull().sum().sum() > 0:
    data.fillna(data.median(), inplace=True)  # Fill missing values with the median of each column

# Encoding the target variable
label_encoder = LabelEncoder()
data['Class'] = label_encoder.fit_transform(data['Class'])  # Encode target labels to numeric values

# Splitting features and target
X = data.drop('Class', axis=1).values  # Features
y = data['Class'].values  # Target

# Standardizing features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # Standardize features for uniform scaling

# Splitting data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


In [54]:
# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)  # Training features
y_train_tensor = torch.tensor(y_train, dtype=torch.long)  # Training labels
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)  # Testing features
y_test_tensor = torch.tensor(y_test, dtype=torch.long)  # Testing labels


In [55]:
# Prepare DataLoader
def create_dataloader(X, y, batch_size):
    dataset = TensorDataset(X, y)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)


In [56]:
# Model Definition
class VanillaMLP(nn.Module):
    def __init__(self, input_size, hidden_layers, activation_fn, num_classes):
        super(VanillaMLP, self).__init__()
        layers = []
        prev_size = input_size
        for hidden_size in hidden_layers:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(activation_fn())
            prev_size = hidden_size
        layers.append(nn.Linear(prev_size, num_classes))  # Output layer
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [57]:
# Hyperparameters
hidden_layer_options = [[4], [8], [16], [32], [64]]  # Varying hidden layers
activation_functions = [nn.Identity, nn.Sigmoid, nn.ReLU, nn.Softmax, nn.Tanh]  # Activation functions
epochs_options = [1, 10, 25, 50, 100, 250]  # Epochs
learning_rates = [10, 1, 0.1, 0.01, 0.001, 0.0001]  # Learning rates
batch_sizes = [16, 32, 64, 128, 256, 512]  # Batch sizes


In [71]:
# Determine number of classes
num_classes = len(np.unique(y))

# Function to run experiments

def experiment_hidden_layers():
    results = []
    hidden_layer_options = [[4], [8], [16], [32], [64]]
    for hidden_layers in hidden_layer_options:
        model = VanillaMLP(X_train.shape[1], hidden_layers, nn.ReLU, num_classes)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        train_loader = create_dataloader(X_train_tensor, y_train_tensor, batch_size=64)
        test_loader = create_dataloader(X_test_tensor, y_test_tensor, batch_size=64)

        for epoch in range(50):
            model.train()
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

        model.eval()
        y_pred = []
        with torch.no_grad():
            for X_batch, _ in test_loader:
                outputs = model(X_batch)
                _, predicted = torch.max(outputs, 1)
                y_pred.extend(predicted.cpu().numpy())

        acc = accuracy_score(y_test, y_pred)
        results.append({"Hidden Layers": hidden_layers, "Accuracy": acc})

    results_df = pd.DataFrame(results)
    print("Experiment Hidden Layers Results:")
    print(results_df)


In [72]:
def experiment_activation_functions():
    results = []
    activation_functions = [nn.Identity, nn.Sigmoid, nn.ReLU, nn.Tanh]
    for activation_fn in activation_functions:
        model = VanillaMLP(X_train.shape[1], [32], activation_fn, num_classes)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        train_loader = create_dataloader(X_train_tensor, y_train_tensor, batch_size=64)
        test_loader = create_dataloader(X_test_tensor, y_test_tensor, batch_size=64)

        for epoch in range(50):
            model.train()
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

        model.eval()
        y_pred = []
        with torch.no_grad():
            for X_batch, _ in test_loader:
                outputs = model(X_batch)
                _, predicted = torch.max(outputs, 1)
                y_pred.extend(predicted.cpu().numpy())

        acc = accuracy_score(y_test, y_pred)
        results.append({"Activation Function": activation_fn.__name__, "Accuracy": acc})

    results_df = pd.DataFrame(results)
    print("Experiment Activation Functions Results:")
    print(results_df)

In [76]:
def experiment_epochs():
    results = []
    epochs_options = [1, 10, 25, 50, 100, 250]
    for epochs in epochs_options:
        model = VanillaMLP(X_train.shape[1], [32], nn.ReLU, num_classes)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        train_loader = create_dataloader(X_train_tensor, y_train_tensor, batch_size=64)
        test_loader = create_dataloader(X_test_tensor, y_test_tensor, batch_size=64)

        for epoch in range(epochs):
            model.train()
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

        model.eval()
        y_pred = []
        with torch.no_grad():
            for X_batch, _ in test_loader:
                outputs = model(X_batch)
                _, predicted = torch.max(outputs, 1)
                y_pred.extend(predicted.cpu().numpy())

        acc = accuracy_score(y_test, y_pred)
        results.append({"Epochs": epochs, "Accuracy": acc})

    results_df = pd.DataFrame(results)
    print("Experiment Epochs Results:")
    print(results_df)



In [74]:
def experiment_learning_rates():
    results = []
    learning_rates = [10, 1, 0.1, 0.01, 0.001, 0.0001]
    for lr in learning_rates:
        model = VanillaMLP(X_train.shape[1], [32], nn.ReLU, num_classes)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        train_loader = create_dataloader(X_train_tensor, y_train_tensor, batch_size=64)
        test_loader = create_dataloader(X_test_tensor, y_test_tensor, batch_size=64)

        for epoch in range(50):
            model.train()
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

        model.eval()
        y_pred = []
        with torch.no_grad():
            for X_batch, _ in test_loader:
                outputs = model(X_batch)
                _, predicted = torch.max(outputs, 1)
                y_pred.extend(predicted.cpu().numpy())

        acc = accuracy_score(y_test, y_pred)
        results.append({"Learning Rate": lr, "Accuracy": acc})

    results_df = pd.DataFrame(results)
    print("Experiment Learning Rates Results:")
    print(results_df)


In [75]:
def experiment_batch_sizes():
    results = []
    batch_sizes = [16, 32, 64, 128, 256, 512]
    for batch_size in batch_sizes:
        model = VanillaMLP(X_train.shape[1], [32], nn.ReLU, num_classes)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        train_loader = create_dataloader(X_train_tensor, y_train_tensor, batch_size=batch_size)
        test_loader = create_dataloader(X_test_tensor, y_test_tensor, batch_size=batch_size)

        for epoch in range(50):
            model.train()
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

        model.eval()
        y_pred = []
        with torch.no_grad():
            for X_batch, _ in test_loader:
                outputs = model(X_batch)
                _, predicted = torch.max(outputs, 1)
                y_pred.extend(predicted.cpu().numpy())

        acc = accuracy_score(y_test, y_pred)
        results.append({"Batch Size": batch_size, "Accuracy": acc})

    results_df = pd.DataFrame(results)
    print("Experiment Batch Sizes Results:")
    print(results_df)

# Run experiments
experiment_hidden_layers()
experiment_activation_functions()
experiment_epochs()
experiment_learning_rates()
experiment_batch_sizes()

print("All experiments completed. Results displayed.")


Experiment Hidden Layers Results:
  Hidden Layers  Accuracy
0           [4]  0.314286
1           [8]  0.600000
2          [16]  0.600000
3          [32]  0.657143
4          [64]  0.542857
Experiment Activation Functions Results:
  Activation Function  Accuracy
0            Identity  0.285714
1             Sigmoid  0.514286
2                ReLU  0.514286
3                Tanh  0.657143
Experiment Epochs Results:
   Epochs  Accuracy
0       1  0.714286
1      10  0.571429
2      25  0.714286
3      50  0.542857
4     100  0.628571
5     250  0.485714
Experiment Learning Rates Results:
   Learning Rate  Accuracy
0        10.0000  0.628571
1         1.0000  0.600000
2         0.1000  0.514286
3         0.0100  0.600000
4         0.0010  0.628571
5         0.0001  0.571429
Experiment Batch Sizes Results:
   Batch Size  Accuracy
0          16  0.571429
1          32  0.514286
2          64  0.600000
3         128  0.657143
4         256  0.571429
5         512  0.628571
All experiments co